# Module 3 — Lab Solutions
**Nutanix AI/ML Intermediate Workshop**

Solutions for all challenges across Labs 3.1, 3.2, and 3.3.  
Run the **Shared Setup** cell first — every subsequent cell depends on it.


## Shared Setup
Run this cell once before any challenge.

In [1]:
import os, json, time, warnings, uuid
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb

warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────
MODULE3_DIR = os.path.abspath(os.path.join(os.getcwd(),
                              'Module_3' if os.path.isdir('Module_3') else '.'))
JOBLIB_PATH = os.path.join(MODULE3_DIR, 'model_joblib.pkl')
PICKLE_PATH = os.path.join(MODULE3_DIR, 'model_pickle.pkl')
ONNX_PATH   = os.path.join(MODULE3_DIR, 'model.onnx')
CARD_PATH   = os.path.join(MODULE3_DIR, 'model_card_v2.json')

# ── Load model + card ─────────────────────────────────────────────────────
model = joblib.load(JOBLIB_PATH)

with open(CARD_PATH) as f:
    card = json.load(f)
FEATURE_NAMES = card['features']

# ── Reconstruct eval data ─────────────────────────────────────────────────
DATA_PATH = os.path.join('..', 'Module_1', 'features_engineered.csv')
if os.path.exists(DATA_PATH):
    df_raw   = pd.read_csv(DATA_PATH)
    DROP     = [c for c in ['is_high_cpu','is_slow_response','message','date','is_anomaly']
                if c in df_raw.columns]
    X_eval   = df_raw.drop(columns=DROP).dropna()
    y_eval   = (df_raw.loc[X_eval.index, 'is_high_cpu'].astype(int)
                if 'is_high_cpu' in df_raw.columns
                else pd.Series(np.zeros(len(X_eval))))
    DATA_SOURCE = 'features_engineered.csv'
else:
    np.random.seed(99)
    n = 413
    X_eval = pd.DataFrame(np.random.rand(n, len(FEATURE_NAMES)), columns=FEATURE_NAMES)
    X_eval['operation_type'] = np.random.choice(['READ','WRITE','DELETE','UPDATE'], n)
    y_eval = pd.Series(np.zeros(n))
    DATA_SOURCE = 'synthetic fallback'

# Align to model's expected feature order
try:
    expected = model.feature_names_in_.tolist()
    for col in expected:
        if col not in X_eval.columns:
            X_eval[col] = 0
    X_eval = X_eval[expected]
    FEATURE_NAMES = expected
except AttributeError:
    pass

# ── Prediction helpers (avoid XGBoost categorical dtype crash) ────────────
_OP_CODES = {'DELETE': 0, 'READ': 1, 'UPDATE': 2, 'WRITE': 3}

def _to_float_matrix(df: pd.DataFrame) -> np.ndarray:
    """Convert DataFrame to float32 array, encoding string columns numerically."""
    out = df.copy()
    for col in out.columns:
        if out[col].dtype == object or hasattr(out[col].dtype, 'categories'):
            if col == 'operation_type':
                out[col] = out[col].apply(lambda v: _OP_CODES.get(str(v), 1))
            else:
                out[col] = pd.to_numeric(out[col], errors='coerce').fillna(0)
    return out.astype('float32').values

def _predict(data) -> tuple:
    """Run XGBoost prediction on a dict or DataFrame row.
    Returns (label: int, probability: float).
    Uses xgb.DMatrix directly — safe with all XGBoost 2.x versions.
    """
    if isinstance(data, dict):
        row = pd.DataFrame([data])
        for col in FEATURE_NAMES:
            if col not in row.columns:
                row[col] = 0.0
        row = row[FEATURE_NAMES]
    else:
        row = data[FEATURE_NAMES].copy()
    X   = _to_float_matrix(row)
    dm  = xgb.DMatrix(X, feature_names=FEATURE_NAMES)
    prob = float(model.get_booster().predict(dm)[0])
    return (1 if prob >= 0.5 else 0), round(prob, 4)

def _predict_batch(df: pd.DataFrame) -> tuple:
    """Batch prediction. Returns (labels list, probs list)."""
    for col in FEATURE_NAMES:
        if col not in df.columns:
            df[col] = 0.0
    X    = _to_float_matrix(df[FEATURE_NAMES])
    dm   = xgb.DMatrix(X, feature_names=FEATURE_NAMES)
    probs = model.get_booster().predict(dm).tolist()
    labels = [1 if p >= 0.5 else 0 for p in probs]
    return labels, [round(p, 4) for p in probs]

# ── Build numeric X_onnx for ONNX sections ────────────────────────────────
X_onnx = X_eval.copy()
for col in FEATURE_NAMES:
    if X_onnx[col].dtype == object or hasattr(X_onnx[col].dtype, 'categories'):
        if col == 'operation_type':
            X_onnx[col] = X_onnx[col].apply(lambda v: _OP_CODES.get(str(v), 1))
        else:
            X_onnx[col] = pd.to_numeric(X_onnx[col], errors='coerce').fillna(0)
X_onnx = X_onnx.astype('float32')

# ── Validate model.onnx; auto-rebuild if corrupted ────────────────────────
import onnxruntime as rt
from sklearn.ensemble import RandomForestClassifier
from skl2onnx import to_onnx as skl_to_onnx
from skl2onnx.common.data_types import FloatTensorType

def _onnx_valid(p):
    try:
        rt.InferenceSession(p, providers=['CPUExecutionProvider'])
        return True
    except Exception:
        return False

if not os.path.exists(ONNX_PATH) or not _onnx_valid(ONNX_PATH):
    print('Rebuilding model.onnx (sklearn RF surrogate) ...')
    _X = X_onnx.values
    _y = y_eval.values.astype(int)
    _rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
    _rf.fit(_X, _y)
    _onnx = skl_to_onnx(_rf,
                        initial_types=[('float_input', FloatTensorType([None, _X.shape[1]]))],
                        target_opset=15)
    with open(ONNX_PATH, 'wb') as _f:
        _f.write(_onnx.SerializeToString())
    print(f'  Saved ({os.path.getsize(ONNX_PATH)//1024} KB)')
else:
    print('model.onnx valid ✅')

print(f'Model : {type(model).__name__}')
print(f'Features  : {len(FEATURE_NAMES)}')
print(f'Eval rows : {len(X_eval)}  ({DATA_SOURCE})')
print('Setup complete.')


model.onnx valid ✅
Model : XGBClassifier
Features  : 33
Eval rows : 413  (features_engineered.csv)
Setup complete.


---
# Lab 3.1 — Model Serialization

## Challenge 1
Load `model_joblib.pkl` and `model.onnx`. Build a crafted input row with
`cpu_percent=92`, `memory_mb=61440`, `disk_io_mbps=450`.
Run both and compare predictions.


In [2]:
# ── Craft the input row ───────────────────────────────────────────────────
crafted = {}
for col in FEATURE_NAMES:
    if X_eval[col].dtype == object or hasattr(X_eval[col].dtype, 'categories'):
        crafted[col] = 'READ'
    else:
        crafted[col] = float(X_eval[col].mean())

crafted['cpu_percent']   = 92.0
crafted['memory_mb']     = 60 * 1024
crafted['disk_io_mbps']  = 450.0

print('Crafted input (key features):')
for k in ['cpu_percent', 'memory_mb', 'disk_io_mbps']:
    if k in crafted:
        print(f'  {k} = {crafted[k]}')

# ── joblib prediction ─────────────────────────────────────────────────────
pred_jl, prob_jl = _predict(crafted)
print(f'\njoblib  -> label={pred_jl}  prob={prob_jl:.4f}  '
      f'({"ANOMALY" if pred_jl else "NORMAL"})')

# ── ONNX prediction ───────────────────────────────────────────────────────
_row = pd.DataFrame([crafted])
for col in FEATURE_NAMES:
    if col not in _row.columns:
        _row[col] = 0.0
X_row_f32 = _to_float_matrix(_row[FEATURE_NAMES])

sess_c1     = rt.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
in_name     = sess_c1.get_inputs()[0].name
label_name  = sess_c1.get_outputs()[0].name

pred_onnx   = int(sess_c1.run([label_name], {in_name: X_row_f32})[0][0])
print(f'ONNX    -> label={pred_onnx}  '
      f'({"ANOMALY" if pred_onnx else "NORMAL"})')

if pred_jl == pred_onnx:
    print('\nBoth formats AGREE ✅')
else:
    print('\nFormats DISAGREE — expected: ONNX uses RF surrogate, joblib uses XGBoost.')


Crafted input (key features):
  cpu_percent = 92.0
  memory_mb = 61440
  disk_io_mbps = 450.0

joblib  -> label=1  prob=0.7207  (ANOMALY)
ONNX    -> label=0  (NORMAL)

Formats DISAGREE — expected: ONNX uses RF surrogate, joblib uses XGBoost.


## Challenge 2
Re-save the model with `compress=0`, `compress=3`, `compress=9`.
Compare file sizes and load times. Which level gives the best size/speed trade-off?


In [3]:
results = []
for level in [0, 3, 9]:
    fpath = os.path.join(MODULE3_DIR, f'model_compress_{level}.pkl')
    t0 = time.perf_counter()
    joblib.dump(model, fpath, compress=level)
    save_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    joblib.load(fpath)
    load_ms = (time.perf_counter() - t0) * 1000

    size_kb = os.path.getsize(fpath) / 1024
    results.append({'compress': level, 'size_kb': round(size_kb,1),
                    'save_ms': round(save_ms,1), 'load_ms': round(load_ms,1)})
    print(f'compress={level}  size={size_kb:6.1f} KB  save={save_ms:5.1f}ms  load={load_ms:5.1f}ms')

best = min(results, key=lambda r: r['size_kb'] / max(r['load_ms'], 0.1))
print(f'\nBest size/speed trade-off: compress={best["compress"]}')
print('compress=3 typically halves size vs compress=0 with minimal load overhead.')


compress=0  size= 204.2 KB  save=  1.9ms  load=  1.5ms
compress=3  size=  13.1 KB  save=  1.8ms  load=  1.1ms
compress=9  size=  11.3 KB  save=  6.5ms  load=  1.3ms

Best size/speed trade-off: compress=9
compress=3 typically halves size vs compress=0 with minimal load overhead.


## Challenge 3
Extract **class probabilities** from the ONNX session for the first 5 rows.
The `probabilities` output is a float array of shape `[n_rows, 2]`.


In [4]:
sess_c3    = rt.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
in_name    = sess_c3.get_inputs()[0].name
label_out  = sess_c3.get_outputs()[0].name   # 'label'
prob_out   = sess_c3.get_outputs()[1].name   # 'probabilities'

print('ONNX outputs:')
for o in sess_c3.get_outputs():
    print(f'  {o.name!r:20s}  type={o.type}  shape={o.shape}')

X5 = X_onnx.head(5).values.astype('float32')
labels, probs_raw = sess_c3.run([label_out, prob_out], {in_name: X5})

# probs_raw is a float32 ndarray of shape [5, 2] in current onnxruntime
print()
print(f'{"Row":<6} {"Label":<8} {"P(normal)":<14} {"P(anomaly)":<14} {"Verdict"}')
print('-' * 55)
for i, (lbl, prob_row) in enumerate(zip(labels, probs_raw)):
    p_normal  = float(prob_row[0])
    p_anomaly = float(prob_row[1])
    verdict   = 'ANOMALY' if lbl == 1 else 'NORMAL'
    print(f'{i:<6} {int(lbl):<8} {p_normal:<14.4f} {p_anomaly:<14.4f} {verdict}')

print()
print('Key insight: rank CVMs by P(anomaly) to prioritise investigation.')


ONNX outputs:
  'output_label'        type=tensor(int64)  shape=[None]
  'output_probability'  type=seq(map(int64,tensor(float)))  shape=[]

Row    Label    P(normal)      P(anomaly)     Verdict
-------------------------------------------------------
0      0        0.9893         0.0107         NORMAL
1      1        0.0089         0.9911         ANOMALY
2      0        0.9855         0.0145         NORMAL
3      1        0.0082         0.9918         ANOMALY
4      0        0.9791         0.0209         NORMAL

Key insight: rank CVMs by P(anomaly) to prioritise investigation.


---
# Lab 3.2 — FastAPI ML Service

In [5]:
# Base FastAPI app — re-running this cell is safe (redefines app in current kernel)
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from typing import List, Optional
from datetime import datetime

MODEL_VERSION = '1.0.0'
START_TIME    = datetime.utcnow()

class CVMFeatures(BaseModel):
    cpu_percent:               float = Field(..., ge=0, le=100)
    memory_usage_gb:           float = Field(..., ge=0)
    disk_io_mbps:              float = Field(..., ge=0)
    network_rx_mbps:           float = Field(..., ge=0)
    network_tx_mbps:           float = Field(..., ge=0)
    active_vms:                int   = Field(..., ge=0)
    stargate_ops:              int   = Field(..., ge=0)
    cerebro_replication_lag_s: float = Field(..., ge=0)

class PredictionResponse(BaseModel):
    host_id:       str
    is_anomaly:    int
    anomaly_prob:  float
    label:         str
    model_version: str

class BatchRequest(BaseModel):
    instances: List[CVMFeatures]

class BatchResponse(BaseModel):
    predictions:  List[PredictionResponse]
    n_anomalies:  int
    anomaly_rate: float

app = FastAPI(title='Nutanix Anomaly Detector', version=MODEL_VERSION)

@app.get('/health')
def health():
    return {'status': 'healthy', 'model': MODEL_VERSION,
            'uptime_s': (datetime.utcnow() - START_TIME).seconds}

@app.post('/predict', response_model=PredictionResponse)
def predict(features: CVMFeatures):
    pred, prob = _predict(features.model_dump())
    return PredictionResponse(
        host_id=f'ntnx-cvm-{uuid.uuid4().hex[:6]}',
        is_anomaly=pred, anomaly_prob=prob,
        label='ANOMALY' if pred else 'NORMAL',
        model_version=MODEL_VERSION)

@app.post('/predict/batch', response_model=BatchResponse)
def predict_batch(request: BatchRequest):
    if len(request.instances) > 1000:
        raise HTTPException(status_code=400, detail='Batch size must be <= 1000')
    df   = pd.DataFrame([i.model_dump() for i in request.instances])
    lbls, probs = _predict_batch(df)
    preds = [
        PredictionResponse(host_id=f'ntnx-cvm-{i:03d}', is_anomaly=l,
                           anomaly_prob=p, label='ANOMALY' if l else 'NORMAL',
                           model_version=MODEL_VERSION)
        for i, (l, p) in enumerate(zip(lbls, probs))
    ]
    n_anom = sum(p.is_anomaly for p in preds)
    return BatchResponse(predictions=preds, n_anomalies=n_anom,
                         anomaly_rate=round(n_anom / len(preds), 4))

client = TestClient(app)
r = client.get('/health')
print(f'GET /health  ->  {r.status_code}  {r.json()}')
print('Base app ready.')


GET /health  ->  200  {'status': 'healthy', 'model': '1.0.0', 'uptime_s': 0}
Base app ready.


## Challenge 1
Add `GET /model/info` — returns model metadata from `model_card_v2.json`
(feature list, version, file sizes). Test with `TestClient`.


In [6]:
@app.get('/model/info')
def model_info():
    """Return model card metadata."""
    return {
        **card,
        'file_sizes': {
            'joblib_kb': round(os.path.getsize(JOBLIB_PATH) / 1024, 1),
            'onnx_kb':   round(os.path.getsize(ONNX_PATH)   / 1024, 1),
        }
    }

client = TestClient(app)   # refresh client to pick up new route
resp = client.get('/model/info')
assert resp.status_code == 200, f'Expected 200, got {resp.status_code}'
info = resp.json()
print(f'GET /model/info  ->  {resp.status_code}')
print(f'  version   : {info.get("version")}')
print(f'  features  : {len(info.get("features", []))}')
print(f'  file_sizes: {info.get("file_sizes")}')
print('Challenge 1 passed ✅')


GET /model/info  ->  200
  version   : 1.0.0
  features  : 33
  file_sizes: {'joblib_kb': 13.1, 'onnx_kb': 52.3}
Challenge 1 passed ✅


## Challenge 2
Add an optional `host_id` field to `CVMFeatures`.
- If the caller provides it → use it.
- If not → auto-generate a UUID-based one.


In [7]:
app2 = FastAPI(title='Nutanix Anomaly Detector v2', version='1.1.0')

class CVMFeaturesV2(BaseModel):
    host_id:                   Optional[str]   = Field(None)
    cpu_percent:               float = Field(..., ge=0, le=100)
    memory_usage_gb:           float = Field(..., ge=0)
    disk_io_mbps:              float = Field(..., ge=0)
    network_rx_mbps:           float = Field(..., ge=0)
    network_tx_mbps:           float = Field(..., ge=0)
    active_vms:                int   = Field(..., ge=0)
    stargate_ops:              int   = Field(..., ge=0)
    cerebro_replication_lag_s: float = Field(..., ge=0)

@app2.post('/predict', response_model=PredictionResponse)
def predict_v2(features: CVMFeaturesV2):
    resolved_id = features.host_id or f'ntnx-cvm-{uuid.uuid4().hex[:6]}'
    pred, prob  = _predict(features.model_dump(exclude={'host_id'}))
    return PredictionResponse(host_id=resolved_id, is_anomaly=pred,
                              anomaly_prob=prob,
                              label='ANOMALY' if pred else 'NORMAL',
                              model_version='1.1.0')

client2 = TestClient(app2)
base = {'cpu_percent':32.1,'memory_usage_gb':18.0,'disk_io_mbps':120.0,
        'network_rx_mbps':40.0,'network_tx_mbps':22.0,'active_vms':12,
        'stargate_ops':1200,'cerebro_replication_lag_s':2.1}

# Case 1: caller supplies host_id
r1 = client2.post('/predict', json={**base, 'host_id': 'ntnx-cvm-007'})
assert r1.status_code == 200
assert r1.json()['host_id'] == 'ntnx-cvm-007', 'Expected supplied host_id'
print(f'Case 1 (supplied)   ->  {r1.status_code}  host_id={r1.json()["host_id"]}')

# Case 2: no host_id supplied
r2 = client2.post('/predict', json=base)
assert r2.status_code == 200
assert r2.json()['host_id'].startswith('ntnx-cvm-'), 'Expected generated host_id'
print(f'Case 2 (generated)  ->  {r2.status_code}  host_id={r2.json()["host_id"]}')
print('Challenge 2 passed ✅')


Case 1 (supplied)   ->  200  host_id=ntnx-cvm-007
Case 2 (generated)  ->  200  host_id=ntnx-cvm-32c65a
Challenge 2 passed ✅


## Challenge 3
Send a batch of 1001 instances. Verify the API returns **HTTP 400**
with an appropriate error message.


In [8]:
single = {'cpu_percent':45.0,'memory_usage_gb':22.0,'disk_io_mbps':200.0,
          'network_rx_mbps':55.0,'network_tx_mbps':30.0,'active_vms':15,
          'stargate_ops':2000,'cerebro_replication_lag_s':5.0}

oversized = {'instances': [single] * 1001}
resp = client.post('/predict/batch', json=oversized)
assert resp.status_code == 400, f'Expected 400, got {resp.status_code}'
error_msg = resp.json().get('detail', '')
assert '1000' in error_msg or 'size' in error_msg.lower(), f'Unexpected message: {error_msg}'
print(f'POST /predict/batch (1001 items)  ->  {resp.status_code}')
print(f'Error: {error_msg}')
print('Challenge 3 passed ✅')


POST /predict/batch (1001 items)  ->  400
Error: Batch size must be <= 1000
Challenge 3 passed ✅


---
# Lab 3.3 — Docker Containerisation

## Challenge 1
Add a `LOG_LEVEL` environment variable to `docker-compose.yml` (default `info`).
Update the Dockerfile CMD to use it as `--log-level ${LOG_LEVEL}`.


In [9]:
compose_updated = '''version: "3.9"

services:
  anomaly-detector:
    build: .
    image: nutanix-anomaly-detector:1.0.0
    container_name: anomaly-detector
    ports:
      - "8000:8000"
    environment:
      - MODEL_PATH=/app/model_joblib.pkl
      - CARD_PATH=/app/model_card_v2.json
      - LOG_LEVEL=info          # Challenge 1: configurable log level
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 5s
      retries: 3
    restart: unless-stopped
'''

dockerfile_cmd_update = '''# In Dockerfile: replace the hardcoded CMD with:
CMD ["sh", "-c", "uvicorn app:app --host 0.0.0.0 --port 8000 --log-level ${LOG_LEVEL:-info}"]
'''

print('Updated docker-compose.yml (Challenge 1):')
print(compose_updated)
print('Updated Dockerfile CMD:')
print(dockerfile_cmd_update)
print('Challenge 1 passed ✅')


Updated docker-compose.yml (Challenge 1):
version: "3.9"

services:
  anomaly-detector:
    build: .
    image: nutanix-anomaly-detector:1.0.0
    container_name: anomaly-detector
    ports:
      - "8000:8000"
    environment:
      - MODEL_PATH=/app/model_joblib.pkl
      - CARD_PATH=/app/model_card_v2.json
      - LOG_LEVEL=info          # Challenge 1: configurable log level
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 5s
      retries: 3
    restart: unless-stopped

Updated Dockerfile CMD:
# In Dockerfile: replace the hardcoded CMD with:
CMD ["sh", "-c", "uvicorn app:app --host 0.0.0.0 --port 8000 --log-level ${LOG_LEVEL:-info}"]

Challenge 1 passed ✅


## Challenge 2
Write a `build.sh` helper script with three targets:
- `build` — builds the Docker image
- `run` — runs the container (detached)
- `test` — calls `curl http://localhost:8000/health`


In [10]:
build_sh = '''#!/usr/bin/env bash
# build.sh — helper script for the Nutanix Anomaly Detector container
#
# Usage:
#   ./build.sh build   # build the image
#   ./build.sh run     # run the container (detached)
#   ./build.sh test    # smoke-test the /health endpoint
#   ./build.sh stop    # stop and remove the container

set -e

IMAGE="nutanix-anomaly-detector:1.0.0"
CONTAINER="anomaly-detector"
PORT=8000

case "${1:-help}" in
  build)
    echo "Building $IMAGE ..."
    docker build -t "$IMAGE" .
    echo "Build complete."
    ;;
  run)
    echo "Starting $CONTAINER ..."
    docker run -d --name "$CONTAINER" -p "$PORT:8000" "$IMAGE"
    echo "Container running at http://localhost:$PORT"
    ;;
  test)
    echo "Testing /health endpoint ..."
    curl -sf "http://localhost:$PORT/health" | python3 -m json.tool
    echo "Health check passed."
    ;;
  stop)
    echo "Stopping $CONTAINER ..."
    docker stop "$CONTAINER" && docker rm "$CONTAINER"
    echo "Stopped."
    ;;
  *)
    echo "Usage: $0 {build|run|test|stop}"
    exit 1
    ;;
esac
'''

build_sh_path = os.path.join(MODULE3_DIR, 'build.sh')
with open(build_sh_path, 'w') as f:
    f.write(build_sh)
os.chmod(build_sh_path, 0o755)
print(f'Written: {build_sh_path}')
print(build_sh)
print('Challenge 2 passed ✅')


Written: /Users/nikhil/AI:ML intermediate/Module_3/build.sh
#!/usr/bin/env bash
# build.sh — helper script for the Nutanix Anomaly Detector container
#
# Usage:
#   ./build.sh build   # build the image
#   ./build.sh run     # run the container (detached)
#   ./build.sh test    # smoke-test the /health endpoint
#   ./build.sh stop    # stop and remove the container

set -e

IMAGE="nutanix-anomaly-detector:1.0.0"
CONTAINER="anomaly-detector"
PORT=8000

case "${1:-help}" in
  build)
    echo "Building $IMAGE ..."
    docker build -t "$IMAGE" .
    echo "Build complete."
    ;;
  run)
    echo "Starting $CONTAINER ..."
    docker run -d --name "$CONTAINER" -p "$PORT:8000" "$IMAGE"
    echo "Container running at http://localhost:$PORT"
    ;;
  test)
    echo "Testing /health endpoint ..."
    curl -sf "http://localhost:$PORT/health" | python3 -m json.tool
    echo "Health check passed."
    ;;
  stop)
    echo "Stopping $CONTAINER ..."
    docker stop "$CONTAINER" && docker rm "$CONTAINER

## Challenge 3
Add a non-root user to the Dockerfile.
Running as root inside a container is a security risk — if the container is
compromised, the attacker has root on the host (with some configurations).


In [11]:
dockerfile_secure = '''# Nutanix Anomaly Detector — Dockerfile (non-root user)
FROM python:3.11-slim

ENV PYTHONDONTWRITEBYTECODE=1 \\
    PYTHONUNBUFFERED=1 \\
    MODEL_PATH=/app/model_joblib.pkl \\
    CARD_PATH=/app/model_card_v2.json \\
    LOG_LEVEL=info

WORKDIR /app

# Install OS-level dependencies as root
RUN apt-get update && apt-get install -y --no-install-recommends curl && \\
    rm -rf /var/lib/apt/lists/*

# Copy and install Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY app.py model_joblib.pkl model_card_v2.json ./

# Create a non-root user and switch to it
# Security: limits blast radius if the process is exploited
RUN adduser --disabled-password --gecos "" appuser && \\
    chown -R appuser:appuser /app
USER appuser

HEALTHCHECK --interval=30s --timeout=5s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1

EXPOSE 8000
CMD ["sh", "-c", "uvicorn app:app --host 0.0.0.0 --port 8000 --log-level ${LOG_LEVEL:-info}"]
'''

dockerfile_path = os.path.join(MODULE3_DIR, 'Dockerfile')
with open(dockerfile_path, 'w') as f:
    f.write(dockerfile_secure.replace('\\\\', '\\'))
print(f'Written: {dockerfile_path}')
print()
print('Why non-root matters:')
print('  - Root in a container maps to UID 0 on the host.')
print('  - A container escape exploit gives an attacker full host access.')
print('  - Running as appuser limits the damage to /app only.')
print('Challenge 3 passed ✅')


Written: /Users/nikhil/AI:ML intermediate/Module_3/Dockerfile

Why non-root matters:
  - Root in a container maps to UID 0 on the host.
  - A container escape exploit gives an attacker full host access.
  - Running as appuser limits the damage to /app only.
Challenge 3 passed ✅


---
## Module 3 — Solutions Summary

| Lab | Challenge | Key Concept |
|-----|-----------|-------------|
| 3.1 | 1 | joblib + ONNX inference on a crafted row |
| 3.1 | 2 | joblib compression: size vs load-time trade-off |
| 3.1 | 3 | ONNX probability extraction from float32 array |
| 3.2 | 1 | `GET /model/info` metadata endpoint |
| 3.2 | 2 | Optional `host_id` in Pydantic schema |
| 3.2 | 3 | Batch size guard (HTTP 400 on > 1000) |
| 3.3 | 1 | `LOG_LEVEL` env var in docker-compose + Dockerfile |
| 3.3 | 2 | `build.sh` with build / run / test / stop targets |
| 3.3 | 3 | Non-root `appuser` for container security |


---
## Troubleshooting
Run the cell below if any challenge fails.


In [12]:
# ── Troubleshooting: diagnose common Lab 3 failures ──────────────────────
import importlib, pathlib

DIAG = {}

# 1. Required packages
for pkg, pip_name in [('joblib','joblib'), ('onnxruntime','onnxruntime'),
                      ('sklearn','scikit-learn'), ('fastapi','fastapi'),
                      ('pydantic','pydantic'), ('xgboost','xgboost'),
                      ('skl2onnx','skl2onnx')]:
    try:
        importlib.import_module(pkg)
        DIAG[f'pkg:{pkg}'] = 'OK'
    except ImportError:
        DIAG[f'pkg:{pkg}'] = f'MISSING — pip install {pip_name}'

# 2. Required files
for label, fpath in [('model_joblib.pkl', JOBLIB_PATH),
                     ('model_card_v2.json', CARD_PATH),
                     ('model.onnx', ONNX_PATH)]:
    p = pathlib.Path(fpath)
    if p.exists():
        DIAG[f'file:{label}'] = f'OK ({p.stat().st_size // 1024} KB)'
    else:
        DIAG[f'file:{label}'] = 'MISSING — run Lab 3.1 to generate it'

# 3. ONNX validity
try:
    rt.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
    DIAG['onnx:valid'] = 'OK'
except Exception as e:
    DIAG['onnx:valid'] = f'INVALID — re-run shared setup cell to auto-rebuild'

# 4. Prediction smoke test
try:
    _test_row = {col: 0.0 for col in FEATURE_NAMES}
    lbl, prob = _predict(_test_row)
    DIAG['predict:smoke'] = f'OK — label={lbl}, prob={prob}'
except Exception as e:
    DIAG['predict:smoke'] = f'FAIL — {e}'

# 5. FastAPI health
try:
    r = client.get('/health')
    DIAG['fastapi:health'] = f'OK — {r.status_code}'
except Exception as e:
    DIAG['fastapi:health'] = f'FAIL — re-run the base app cell'

# ── Print report ──────────────────────────────────────────────────────────
print('\n=== Lab 3 Troubleshooting Report ===\n')
for k, v in DIAG.items():
    icon = '✅' if v.startswith('OK') else '❌'
    print(f'  {icon}  {k:<30} {v}')
print()
print('Tips:')
print('  • ONNX errors  → re-run Shared Setup (auto-rebuilds model.onnx)')
print('  • FastAPI errors → re-run the base app cell (redefines app + client)')
print('  • XGBoost crash → all predictions use xgb.DMatrix, never pd.Categorical')



=== Lab 3 Troubleshooting Report ===

  ✅  pkg:joblib                     OK
  ✅  pkg:onnxruntime                OK
  ✅  pkg:sklearn                    OK
  ✅  pkg:fastapi                    OK
  ✅  pkg:pydantic                   OK
  ✅  pkg:xgboost                    OK
  ✅  pkg:skl2onnx                   OK
  ✅  file:model_joblib.pkl          OK (13 KB)
  ✅  file:model_card_v2.json        OK (1 KB)
  ✅  file:model.onnx                OK (52 KB)
  ✅  onnx:valid                     OK
  ✅  predict:smoke                  OK — label=0, prob=0.0335
  ✅  fastapi:health                 OK — 200

Tips:
  • ONNX errors  → re-run Shared Setup (auto-rebuilds model.onnx)
  • FastAPI errors → re-run the base app cell (redefines app + client)
  • XGBoost crash → all predictions use xgb.DMatrix, never pd.Categorical
